# BEP

## Pre-work

In [29]:
# imports

from pathlib import Path
import pandas as pd
import json

In [25]:
# constants

ROOT_DIR = Path.cwd().parent
DATASET_DIR = ROOT_DIR / "datasets"

NOVIELLI_GOLD_STANDARD_DIR = DATASET_DIR / "novielli-gold-standard"
GH_GOLD_STANDARD = NOVIELLI_GOLD_STANDARD_DIR / "Github Gold Standard.csv"
SO_GOLD_STANDARD = NOVIELLI_GOLD_STANDARD_DIR / "Stack Overflow Gold Standard.csv"

COUTINHO_DIR = DATASET_DIR / "coutinho-sentiment-replication"
COUTINHO = COUTINHO_DIR / "dataset.json"

### Data collection (Novielli et al., 2021)

The dataset used by Novieli et al. is in CSV format, which makes it straight forward to create dataframes out of it.

In [23]:
df_gs_gh = pd.read_csv(GH_GOLD_STANDARD)
df_gs_so = pd.read_csv(SO_GOLD_STANDARD)

In [22]:
print("GH Gold Standard")
df_gs_gh

GH Gold Standard


,Type,ID,Text,Gold label
0,commit comments no security,329914,Hmm... Very mysterious development. Not be eas...,negative
1,commit comments no security,2728032,":+1:""",neutral
2,commit comments no security,3098507,So what I'm hearing is that if I: * tag consis...,neutral
3,commit comments no security,3103230,How did you tested by normal player? Have you ...,positive
4,commit comments no security,95688,Nice indeed. An earlier version of this patch ...,positive
...,...,...,...,...
393,PR comments security,739112,Can clarify this with `Hash#fetch`: ``` second...,neutral
394,PR comments security,1217254,It does: Endianess is Netty (and in Java gener...,neutral
395,PR comments security,1581644,In the spirit of testing all the guarantees in...,neutral
396,PR comments security,3991765,More proper would be: ```php if (self::$truste...,neutral


In [20]:
print("SO Gold Standard")
df_gs_so

SO Gold Standard


,ID,Text,Gold label
0,25607311,I saw a really nice post on stackoverflow on t...,positive
1,25386947,I tried so many different ways for printing wi...,negative
2,25419608,I have a general style definition as Now want ...,neutral
3,25727724,My category.php uses wp_list_categories to dis...,neutral
4,25346200,I'm trying to test my controller which deal wi...,neutral
...,...,...,...
196,25420741,I have written a code in java for web services...,negative
197,25695655,Under the code that i'm trying to use for add ...,mixed
198,25614128,"According to the manual, it says . In my opini...",neutral
199,25477108,I am designing a system that will need to deal...,neutral


### Data collection (Coutinho et al., 2024)

Since this dataset is in nested JSON format, we do some extra pre-processing to normalize it.

In [31]:
text = COUTINHO.read_text(encoding="utf-8")

if text.lstrip().startswith("["):
    data = json.loads(text)
else:
    try:
        data = [json.loads(line) for line in text.splitlines() if line.strip()]
    except json.JSONDecodeError:
        data = [json.loads(text)]

df_cou = pd.json_normalize(data, sep="_")
wanted = [
    "raw_message", "clean_message", "message_url",
    "part2_aggregate_polarity", "part2_aggregate_avg_confidence", "part2_aggregate_agreement_type",
    "tools_SentiStrength", "tools_SentiStrengthSE", "tools_SentiCR", "tools_DEVA", "tools_Senti4SD"
]
df_cou = df[[c for c in wanted if c in df.columns]]

In [37]:
print(df_cou.shape)
df_cou

(1791, 11)


,raw_message,clean_message,message_url,part2_aggregate_polarity,part2_aggregate_avg_confidence,part2_aggregate_agreement_type,tools_SentiStrength,tools_SentiStrengthSE,tools_SentiCR,tools_DEVA,tools_Senti4SD
0,@snicoll did you get a chance to follow up on ...,did you get a chance to follow up on the issue...,https://github.com/spring-projects/spring-boot...,neutral,4.333333,neuro_and_comp,neutral,neutral,neutral,neutral,neutral
1,Thank you for this PR. This is an interesting...,Thank you for this PR. This is an interesting...,https://github.com/google/guice/pull/1349#issu...,undefined,4.666667,undefined,negative,negative,negative,negative,neutral
2,"@mbhave, oh sorry for this, my fault, i'll be...",", oh sorry for this, my fault, i'll be more c...",https://github.com/spring-projects/spring-boot...,negative,5.000000,all,negative,negative,neutral,negative,neutral
3,I've added `merge-with-amendments` so that we ...,I've added so that we can add documentation r...,https://github.com/spring-projects/spring-boot...,neutral,4.000000,comp_only,neutral,neutral,neutral,neutral,neutral
4,> @Primary and @ConditionalOnMissingBean. I th...,Applications can still use or qualifiers to g...,https://github.com/spring-projects/spring-boot...,negative,4.666667,all,negative,negative,neutral,negative,negative
...,...,...,...,...,...,...,...,...,...,...,...
1786,"Hmm, looks like r737027 corresponds to version...","Hmm, looks like r737027 corresponds to version...",https://github.com/puppeteer/puppeteer/pull/56...,negative,3.333333,neuro_and_comp,positive,negative,neutral,neutral,positive
1787,"<!--\r\nFirst of all, thank you for your contr...","<!-- First of all, thank you for your contribu...",https://github.com/ant-design/ant-design/pull/...,neutral,4.666667,all,negative,negative,negative,negative,positive
1788,"there was a typo in the comment, also made a s...","there was a typo in the comment, also made a s...",https://github.com/tensorflow/tensorflow/pull/...,neutral,4.000000,all,neutral,neutral,neutral,neutral,neutral
1789,Fixed some typos. Added the ghost button type....,Fixed some typos. Added the ghost button type....,https://github.com/ant-design/ant-design/pull/...,neutral,4.000000,all,neutral,negative,neutral,negative,neutral
